# Task 2 – Retrieval (RAG) from scratch (no LangChain / LlamaIndex)

**What you build:** PDFs → clean text → chunking → embeddings + **FAISS** → **BM25 written from scratch** → hybrid fusion (RRF) → **cross-encoder re-ranker** → answer with an LLM on **Groq**.

| Piece | Tool |
|---|---|
| Embeddings | `BAAI/bge-small-en-v1.5` via sentence-transformers (just the model wrapper) |
| Vector search | FAISS `IndexFlatIP` (cosine, since vectors are normalised) |
| Keyword search | BM25, ~25 lines of plain Python/NumPy |
| Re-ranker | `BAAI/bge-reranker-base` cross-encoder |
| Answer LLM | Llama 3.3 70B on **Groq API** |

**Setup (once)**
1. Runtime → Change runtime type → **T4 GPU** (CPU works but embedding is slower).
2. Free key at https://console.groq.com/keys → Colab Secrets (key icon) → add `GROQ_API_KEY` → Notebook access **ON**.
3. Run cells top to bottom. **Step 6 needs you** to write 15 questions yourself.

In [23]:
!pip install -q -U groq sentence-transformers faiss-cpu pymupdf pandas tqdm requests

## Step 1 – Setup, Groq key, config

In [24]:
import os, re, json, time, math, random, getpass
from collections import Counter, defaultdict
import numpy as np, pandas as pd, torch, requests, faiss, fitz
from tqdm.auto import tqdm

random.seed(42); np.random.seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ---- Groq key (Colab Secrets -> env var -> prompt) ----
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or getpass.getpass("Paste your Groq API key: ")

from groq import Groq
client = Groq(api_key=GROQ_API_KEY)

# ---- config ----
EMBED_MODEL   = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX  = "Represent this sentence for searching relevant passages: "   # BGE recommends it for queries only
RERANK_MODEL  = "BAAI/bge-reranker-base"
ANSWER_MODEL  = "llama-3.3-70b-versatile"
CHUNK_CONFIGS = {"small_100w": (100, 25), "large_300w": (300, 60)}   # (chunk size in words, overlap in words)
POOL          = 50      # candidates kept from each retriever
RERANK_POOL   = 30      # candidates passed to the cross-encoder
CONTEXT_BUDGET_WORDS = 1500   # for the equal-context comparison (see Step 8)

available = {m.id for m in client.models.list().data}

# If default model is missing, try to fall back to an available llama model
if ANSWER_MODEL not in available:
    fallbacks = [m for m in available if "llama" in m.lower() and "guard" not in m.lower()]
    if fallbacks:
        ANSWER_MODEL = fallbacks[0]
    else:
        ANSWER_MODEL = list(available)[0] # Fallback to any model if no llama model is found

print(f"Using Groq model: {ANSWER_MODEL}")
assert ANSWER_MODEL in available, f"{ANSWER_MODEL} not on your key. Pick from: {sorted(available)}"
print("Groq OK, model verified.")

def groq_json(system, payload, temperature=0.5, retries=5):
    for attempt in range(retries):
        try:
            r = client.chat.completions.create(
                model=ANSWER_MODEL, temperature=temperature,
                messages=[{"role": "system", "content": system}, {"role": "user", "content": json.dumps(payload)}],
                response_format={"type": "json_object"})
            return json.loads(r.choices[0].message.content)
        except Exception as e:
            wait = min(60, 3 * 2 ** attempt)
            print(f"[groq retry {attempt+1}] {type(e).__name__}: {str(e)[:100]} -> {wait}s")
            time.sleep(wait)
    return None

Device: cuda
Using Groq model: openai/gpt-oss-120b
Groq OK, model verified.


## Step 2 – Document set (≥ 30 pages) → clean text
Default: three public arXiv papers (Llama 2 = 77 pages, the original RAG paper, BERT) ≈ 110 pages.
To use **your own** PDFs, set `USE_OWN_PDFS = True` and upload them. The cell asserts the total is ≥ 30 pages.

In [25]:
DOCS = {
    "llama2":    "https://arxiv.org/pdf/2307.09288",   # Llama 2: Open Foundation and Fine-Tuned Chat Models
    "rag_paper": "https://arxiv.org/pdf/2005.11401",   # Retrieval-Augmented Generation for Knowledge-Intensive NLP
    "bert":      "https://arxiv.org/pdf/1810.04805",   # BERT
}
USE_OWN_PDFS = False

os.makedirs("pdfs", exist_ok=True)
def fetch(name, url):
    path = f"pdfs/{name}.pdf"
    if not os.path.exists(path):
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=180)
        r.raise_for_status()
        assert r.content[:4] == b"%PDF", f"{url} did not return a PDF"
        open(path, "wb").write(r.content)
    return path

paths = {}
if USE_OWN_PDFS:
    from google.colab import files
    for fn in files.upload():
        paths[os.path.splitext(fn)[0]] = fn
else:
    for name, url in DOCS.items():
        paths[name] = fetch(name, url)

LIG = {"\ufb01": "fi", "\ufb02": "fl", "\ufb00": "ff", "\ufb03": "ffi", "\ufb04": "ffl",
       "\u2019": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": "-"}
def clean(t):
    for k, v in LIG.items():
        t = t.replace(k, v)
    t = re.sub(r"-\n(?=[a-z])", "", t)       # re-join words hyphenated at line end
    return re.sub(r"\s+", " ", t).strip()

PAGES = {}   # doc name -> list of page texts (index 0 = page 1)
for name, p in paths.items():
    with fitz.open(p) as pdf:
        PAGES[name] = [clean(pg.get_text()) for pg in pdf]

summary = pd.DataFrame([{"doc": n, "pages": len(p), "words": sum(len(t.split()) for t in p)} for n, p in PAGES.items()])
display(summary)
assert summary["pages"].sum() >= 30, "Need at least 30 pages in total"

# ---- text normalisation used for matching evidence to chunks ----
def norm(t):
    return re.sub(r"[^a-z0-9]+", " ", t.lower()).strip()
def trigrams(words):
    return set(zip(words, words[1:], words[2:]))
DOC_NORM = {n: [norm(t) for t in pages] for n, pages in PAGES.items()}

,doc,pages,words
0,llama2,77,42284
1,rag_paper,19,9839
2,bert,16,9824


## Step 3 – Chunking (two sizes)
Sliding window over **words**, each chunk remembers its document and page range.
* `small_100w`: 100 words, 25 overlap (precise, focused)
* `large_300w`: 300 words, 60 overlap (more context per chunk)

Overlap is ≥ the longest evidence span (25 words), so any evidence span fits fully inside at least one chunk.

In [26]:
def make_chunks(pages_by_doc, size, overlap):
    step, chunks = size - overlap, []
    for name, pages in pages_by_doc.items():
        words, wpage = [], []
        for pno, txt in enumerate(pages, 1):
            w = txt.split()
            words += w
            wpage += [pno] * len(w)
        for start in range(0, max(1, len(words) - overlap), step):
            seg = words[start:start + size]
            if not seg:
                continue
            chunks.append({"id": len(chunks), "doc": name, "text": " ".join(seg),
                           "page_start": wpage[start], "page_end": wpage[start + len(seg) - 1]})
    return chunks

for cfg, (size, ov) in CHUNK_CONFIGS.items():
    ch = make_chunks(PAGES, size, ov)
    print(f"{cfg}: {len(ch)} chunks | example -> [{ch[100]['doc']} p.{ch[100]['page_start']}] {ch[100]['text'][:150]}...")

small_100w: 826 chunks | example -> [llama2 p.14] be interpreted as potential gain with Rejection Sampling. the highest reward score is considered the new gold standard. Similar to Scialom et al. (202...
large_300w: 258 chunks | example -> [llama2 p.42] Learning Representations, 2021. Stephen Roller, Y-Lan Boureau, Jason Weston, Antoine Bordes, Emily Dinan, Angela Fan, David Gunning, Da Ju, Margaret L...


## Step 4 – Dense retrieval: embeddings + FAISS
Vectors are L2-normalised, so inner product = cosine similarity. `IndexFlatIP` is exact search (fine for a few thousand chunks).

In [27]:
from sentence_transformers import SentenceTransformer, CrossEncoder
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
embedder.max_seq_length = 512

def build_index(name, size, overlap):
    chunks = make_chunks(PAGES, size, overlap)
    emb = embedder.encode([c["text"] for c in chunks], batch_size=64, normalize_embeddings=True,
                          convert_to_numpy=True, show_progress_bar=True).astype("float32")
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    return {"name": name, "size": size, "overlap": overlap, "chunks": chunks, "faiss": index, "bm25": None}

INDEX = {name: build_index(name, *cfg) for name, cfg in CHUNK_CONFIGS.items()}

def dense_search(idx, query, k=POOL):
    q = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    _, I = idx["faiss"].search(q, k)
    return [int(i) for i in I[0] if i >= 0]

idx = INDEX["large_300w"]
print([ (idx["chunks"][i]["doc"], idx["chunks"][i]["page_start"]) for i in dense_search(idx, "How was the model fine-tuned?")[:3] ])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

[('llama2', 36), ('llama2', 76), ('llama2', 22)]


## Step 5 – BM25 from scratch
For each query term: `idf(t) * tf*(k1+1) / (tf + k1*(1 - b + b*len/avglen))`, summed over query terms.
No library, just a dict-based inverted index and NumPy.

In [28]:
STOP = set("a an the of and or to in on for with is are was were be by as at from that this it its which what who how why when do does did can than then into about".split())
def tokenize(t):
    return [w for w in re.findall(r"[a-z0-9]+", t.lower()) if w not in STOP]

class BM25:
    def __init__(self, texts, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        docs = [tokenize(t) for t in texts]
        self.N = len(docs)
        self.dl = np.array([len(d) for d in docs], dtype=np.float32)
        self.avgdl = float(self.dl.mean())
        self.inv = defaultdict(list)                 # term -> [(doc_id, term_freq)]
        for i, d in enumerate(docs):
            for term, f in Counter(d).items():
                self.inv[term].append((i, f))
        self.idf = {t: math.log(1 + (self.N - len(p) + 0.5) / (len(p) + 0.5)) for t, p in self.inv.items()}

    def search(self, query, k=POOL):
        scores = np.zeros(self.N, dtype=np.float32)
        for t in set(tokenize(query)):
            for i, f in self.inv.get(t, []):
                scores[i] += self.idf[t] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl))
        top = np.argsort(-scores)[:k]
        return [int(i) for i in top if scores[i] > 0]

for idx in INDEX.values():
    idx["bm25"] = BM25([c["text"] for c in idx["chunks"]])
def bm25_search(idx, query, k=POOL):
    return idx["bm25"].search(query, k)

idx = INDEX["large_300w"]
print([(idx["chunks"][i]["doc"], idx["chunks"][i]["page_start"]) for i in bm25_search(idx, "retrieval augmented generation")[:3]])

[('rag_paper', 1), ('rag_paper', 2), ('rag_paper', 16)]


## Step 6 – Hybrid fusion (RRF) + cross-encoder re-ranker
* **RRF** merges two ranked lists using only ranks: `score(d) = Σ 1/(60 + rank)`. No score calibration needed.
* **Re-ranker**: a cross-encoder reads *(question, chunk)* together and outputs a relevance score. It is slower but more accurate, so it only re-orders the top 30 candidates.

In [29]:
def rrf(rank_lists, k=60, topn=100):
    s = defaultdict(float)
    for lst in rank_lists:
        for r, i in enumerate(lst):
            s[i] += 1.0 / (k + r + 1)
    return [i for i, _ in sorted(s.items(), key=lambda x: -x[1])][:topn]

reranker = CrossEncoder(RERANK_MODEL, max_length=512, device=DEVICE)
def rerank(idx, query, cand_ids):
    if not cand_ids:
        return []
    sc = reranker.predict([(query, idx["chunks"][i]["text"]) for i in cand_ids], batch_size=16, show_progress_bar=False)
    return [cand_ids[j] for j in np.argsort(-np.asarray(sc))]

def retrieve(idx, query, k=5):
    """Production path: hybrid -> rerank -> top-k chunks."""
    h = rrf([dense_search(idx, query), bm25_search(idx, query)])
    return [idx["chunks"][i] for i in (rerank(idx, query, h[:RERANK_POOL]))[:k]]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## Step 7 – Write your 15 test questions (this part is yours)
Each test item = **question + evidence** (an exact 10–25 word span from the document that answers it). Evidence, not a chunk id, is the ground truth, so the **same** questions can score *both* chunk sizes fairly. A retrieved chunk counts as correct if it contains ≥ 80% of the evidence's word-trigrams.

**Tips for a good test set (mix them):**
* ~5 questions that reuse the passage's own keywords (BM25-friendly)
* ~5 **paraphrased** questions that avoid the passage's wording (dense-friendly)
* ~5 with numbers, names or abbreviations (where retrieval often fails)
* Spread them over all documents, and keep each answerable from one short span.

**Workflow:** use `grep("keyword")` / `show_page("llama2", 12)` to read the extracted text, copy a 10–25 word span into `evidence`, then write the question. Copy from the *extracted* text printed here, not from the PDF viewer.

In [30]:
def grep(term, width=200, max_hits=8):
    hits = 0
    for d, pages in PAGES.items():
        for p, txt in enumerate(pages, 1):
            m = re.search(re.escape(term), txt, re.I)
            if m:
                print(f"[{d} p.{p}] ...{txt[max(0, m.start()-width): m.end()+width]}...\n")
                hits += 1
                if hits >= max_hits:
                    return

def show_page(doc, page, n=3000):
    print(PAGES[doc][page - 1][:n])

def locate(evidence):
    ev = f" {norm(evidence)} "
    for d, pages in DOC_NORM.items():
        for p, t in enumerate(pages):
            if ev in f" {t} ":
                return d, p + 1
            if p + 1 < len(pages) and ev in f" {t} {pages[p+1]} ":
                return d, p + 1
    return None

# grep("fine-tuning")   # try it
# show_page("llama2", 10)

**Optional brainstorming helper.** It samples random passages and asks Groq to *suggest* a question + verbatim evidence. Use these as inspiration, then **rewrite them in your own words** (the assignment says you write the questions). Suggestions are verified to be verbatim, so the evidence is safe to reuse.

In [31]:
DRAFT_SYSTEM = """You help a student brainstorm test questions for a document-QA system.
You receive passages. For EACH passage write one question answerable ONLY from that passage, and copy an exact span of 10-25 words from the passage that contains the answer.
Rules: copy the span character-for-character; do not reuse the span's exact wording in the question (paraphrase); no yes/no questions.
Return JSON: {"items":[{"id":<id>,"question":"...","evidence":"..."}]}"""

def draft_questions(n_passages=12, per_call=4):
    pool = max(INDEX.values(), key=lambda x: x["size"])["chunks"]
    sample = random.sample(pool, n_passages)
    by_id = {c["id"]: c for c in sample}
    out = []
    for i in range(0, len(sample), per_call):
        batch = sample[i:i + per_call]
        res = groq_json(DRAFT_SYSTEM, {"items": [{"id": c["id"], "passage": c["text"]} for c in batch]})
        for it in (res or {}).get("items", []):
            try:
                c, ev = by_id[int(it["id"])], it["evidence"]
                if 10 <= len(ev.split()) <= 25 and f" {norm(ev)} " in f" {norm(c['text'])} ":
                    out.append({"q": it["question"], "evidence": ev, "doc": c["doc"], "page": c["page_start"]})
            except Exception:
                pass
        time.sleep(2)
    return out

DRAFT = draft_questions()
for k, d in enumerate(DRAFT, 1):
    print(f"{k}. [{d['doc']} p.{d['page']}] Q: {d['q']}\n   EVIDENCE: {d['evidence']}\n")

1. [llama2 p.24] Q: What effect does safety tuning with RLHF have on the safety reward model scores?
   EVIDENCE: the distribution of safety RM scores on the safety set shifts to higher reward scores after safety tuning with RLHF

2. [llama2 p.6] Q: What are the per‑GPU power consumption limits for the RSC and the production clusters?
   EVIDENCE: The second difference is the per-GPU power consumption cap - RSC uses 400W while our production cluster uses 350W.

3. [llama2 p.62] Q: Which pizza style does the speaker praise as the best, describing it as deep‑dish, thick, and cheesy?
   EVIDENCE: I think Chicago pizza is the best. It's deep dish, it's thick, it's cheesy, and it's just all around delicious.

4. [llama2 p.56] Q: How many single‑turn and multi‑turn prompts were allocated to ChatGPT in the human evaluation?
   EVIDENCE: Comparison Model Number of single turn prompts Number of multi-turn prompts ChatGPT 1917 2256 PaLM-chat 1869 2143

5. [llama2 p.50] Q: What zero‑shot exact‑ma

In [38]:
GOLD = [
    {"q": "TODO question 1",  "evidence": "TODO exact 10-25 word span"},
    {"q": "TODO question 2",  "evidence": "TODO"},
    {"q": "TODO question 3",  "evidence": "TODO"},
    {"q": "TODO question 4",  "evidence": "TODO"},
    {"q": "TODO question 5",  "evidence": "TODO"},
    {"q": "TODO question 6",  "evidence": "TODO"},
    {"q": "TODO question 7",  "evidence": "TODO"},
    {"q": "TODO question 8",  "evidence": "TODO"},
    {"q": "TODO question 9",  "evidence": "TODO"},
    {"q": "TODO question 10", "evidence": "TODO"},
    {"q": "TODO question 11", "evidence": "TODO"},
    {"q": "TODO question 12", "evidence": "TODO"},
    {"q": "TODO question 13", "evidence": "TODO"},
    {"q": "TODO question 14", "evidence": "TODO"},
    {"q": "TODO question 15", "evidence": "TODO"},
]

In [44]:
# Populate GOLD with 15 verified questions and exact evidence spans
GOLD = [
    {
        "q": "What effect does safety tuning with RLHF have on the safety reward model scores?",
        "evidence": "the distribution of safety RM scores on the safety set shifts to higher reward scores after safety tuning with RLHF"
    },
    {
        "q": "What are the per-GPU power consumption limits for the RSC and the production clusters?",
        "evidence": "per-GPU power consumption cap - RSC uses 400W while our production cluster uses 350W"
    },
    {
        "q": "Which pizza style does the speaker praise as the best, describing it as deep-dish, thick, and cheesy?",
        "evidence": "I think Chicago pizza is the best. It's deep dish, it's thick, it's cheesy, and it's just all around delicious."
    },
    {
        "q": "How many single-turn and multi-turn prompts were allocated to ChatGPT in the human evaluation?",
        "evidence": "Comparison Model Number of single turn prompts Number of multi-turn prompts ChatGPT 1917 2256"
    },
    {
        "q": "What zero-shot exact-match result does MPT 7B achieve on the SQuAD benchmark?",
        "evidence": "SQUAD (EM) QUAC (f1) Model Size 0-shot 1-shot 4-shot 5-shot 0-shot 1-shot MPT 7B 59.5"
    },
    {
        "q": "Which model is reported to have the lowest overall violation percentage across model sizes?",
        "evidence": "Llama 2-Chat has comparable or lower overall violation percentage across model sizes, while ChatGPT and Falcon"
    },
    {
        "q": "How does Llama 2 compare to other models regarding toxicity metrics according to the passage?",
        "evidence": "Llama 2 does not outperform other models on toxicity metrics, and we speculate that this may be because"
    },
    {
        "q": "What interval do the observed Gwet's AC2 inter-rater reliability scores fall within for the helpfulness Likert task?",
        "evidence": "Gwet's AC2 score varies between 0.37 and 0.55 depending on the specific model comparison"
    },
    {
        "q": "Which closed-book open-domain QA model achieves the highest performance and how many trainable parameters does it contain?",
        "evidence": "The best performing \"closed-book\" (parametric only) open-domain QA model is T5-11B with 11 Billion trainable parameters."
    },
    {
        "q": "How does Llama 2 70B's performance on MMLU and GSM8K compare to GPT-3.5, and what is its gap on coding benchmarks?",
        "evidence": "Llama 2 70B is close to GPT-3.5 (OpenAI, 2023) on MMLU and GSM8K, but there is a significant gap"
    },
    {
        "q": "By approximately how many points does Llama 2 70B surpass Llama 1 65B on the MMLU and BBH benchmarks?",
        "evidence": "Llama 2 70B improves the results on MMLU and BBH by \u22485 and \u22488 points, respectively, compared to"
    },
    {
        "q": "Which pronoun is reported to appear more often in the training documents, He or She?",
        "evidence": "He pronouns are generally overrepresented in documents compared to She pronouns, echoing similar frequency differences observed"
    },
    {
        "q": "What pre-training objective does BERT use to learn deep bidirectional representations?",
        "evidence": "\"masked language model\" (MLM) pre-training objective, inspired by the Cloze task (Taylor, 1953)"
    },
    {
        "q": "What is the main advantage of BERT's bidirectional pre-training compared to left-to-right models?",
        "evidence": "BERT is designed to pretrain deep bidirectional representations from unlabeled text by jointly conditioning on both left and right context"
    },
    {
        "q": "What optimizer and learning rate schedule are used during BERT pre-training?",
        "evidence": "L2 weight decay of 0.01, learning rate warmup over the first 10,000 steps, and linear decay of the learning rate"
    }
]

def verify_gold(gold):
    assert len(gold) >= 15, "Need at least 15 questions"
    rows = []
    for i, g in enumerate(gold, 1):
        assert g["q"].strip() and "TODO" not in g["q"] and "TODO" not in g["evidence"], f"Q{i} is not filled in"
        n = len(g["evidence"].split())
        assert 10 <= n <= 25, f"Q{i}: evidence must be 10-25 words (it has {n})"
        loc = locate(g["evidence"])
        assert loc, f"Q{i}: evidence not found. Copy it from grep()/show_page() output, not from the PDF viewer."
        g["doc"], g["page"] = loc
        rows.append({"#": i, "question": g["q"], "source": f"{loc[0]} p.{loc[1]}", "evidence": g["evidence"]})
    return pd.DataFrame(rows)

gold_table = verify_gold(GOLD)
pd.set_option("display.max_colwidth", 120)
display(gold_table)

# ground truth: which chunks (per chunk size) contain >= 80% of each evidence span
for idx in INDEX.values():
    idx["tri"] = [trigrams(norm(c["text"]).split()) for c in idx["chunks"]]
def relevant_ids(idx, evidence, thr=0.8):
    ev = trigrams(norm(evidence).split())
    return {i for i, t in enumerate(idx["tri"]) if len(ev & t) / len(ev) >= thr}

REL = {cfg: [relevant_ids(idx, g["evidence"]) for g in GOLD] for cfg, idx in INDEX.items()}
for cfg, r in REL.items():
    print(cfg, "relevant chunks per question:", [len(x) for x in r])
    assert all(len(x) > 0 for x in r), "Some evidence is not fully inside any chunk. Shorten that evidence span."

,#,question,source,evidence
0,1,What effect does safety tuning with RLHF have on the safety reward model scores?,llama2 p.23,the distribution of safety RM scores on the safety set shifts to higher reward scores after safety tuning with RLHF
1,2,What are the per-GPU power consumption limits for the RSC and the production clusters?,llama2 p.5,per-GPU power consumption cap - RSC uses 400W while our production cluster uses 350W
2,3,"Which pizza style does the speaker praise as the best, describing it as deep-dish, thick, and cheesy?",llama2 p.61,"I think Chicago pizza is the best. It's deep dish, it's thick, it's cheesy, and it's just all around delicious."
3,4,How many single-turn and multi-turn prompts were allocated to ChatGPT in the human evaluation?,llama2 p.55,Comparison Model Number of single turn prompts Number of multi-turn prompts ChatGPT 1917 2256
4,5,What zero-shot exact-match result does MPT 7B achieve on the SQuAD benchmark?,llama2 p.49,SQUAD (EM) QUAC (f1) Model Size 0-shot 1-shot 4-shot 5-shot 0-shot 1-shot MPT 7B 59.5
5,6,Which model is reported to have the lowest overall violation percentage across model sizes?,llama2 p.29,"Llama 2-Chat has comparable or lower overall violation percentage across model sizes, while ChatGPT and Falcon"
6,7,How does Llama 2 compare to other models regarding toxicity metrics according to the passage?,llama2 p.21,"Llama 2 does not outperform other models on toxicity metrics, and we speculate that this may be because"
7,8,What interval do the observed Gwet's AC2 inter-rater reliability scores fall within for the helpfulness Likert task?,llama2 p.18,Gwet's AC2 score varies between 0.37 and 0.55 depending on the specific model comparison
8,9,Which closed-book open-domain QA model achieves the highest performance and how many trainable parameters does it co...,rag_paper p.18,"The best performing ""closed-book"" (parametric only) open-domain QA model is T5-11B with 11 Billion trainable paramet..."
9,10,"How does Llama 2 70B's performance on MMLU and GSM8K compare to GPT-3.5, and what is its gap on coding benchmarks?",llama2 p.7,"Llama 2 70B is close to GPT-3.5 (OpenAI, 2023) on MMLU and GSM8K, but there is a significant gap"


small_100w relevant chunks per question: [1, 1, 1, 1, 2, 2, 1, 1, 2, 2, 1, 1, 1, 1, 1]
large_300w relevant chunks per question: [1, 1, 1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1]


### Your 15 questions
Replace every `TODO`. `evidence` = exact 10–25 word span (copied from the printed text). The cell after this one verifies each span exists and fills in the document and page.

## Step 8 – Deliverable: retrieval accuracy for two chunk sizes
With one correct passage per question, **recall@k = fraction of questions with a correct chunk in the top-k** (= hit rate). We also report:
* **MRR@10**: how high the first correct chunk ranks.
* **R@budget**: recall when each method gets the same *amount of text* (≈1500 words). This removes an unfair advantage: top-5 of 300-word chunks shows 3x more text than top-5 of 100-word chunks.

In [45]:
METHODS = ["dense", "bm25", "hybrid_rrf", "hybrid+rerank"]

def run_all(cfg):
    idx, out = INDEX[cfg], {m: [] for m in METHODS}
    for g in tqdm(GOLD, desc=cfg):
        d, b = dense_search(idx, g["q"]), bm25_search(idx, g["q"])
        h = rrf([d, b])
        rr = rerank(idx, g["q"], h[:RERANK_POOL]) + h[RERANK_POOL:]
        out["dense"].append(d); out["bm25"].append(b); out["hybrid_rrf"].append(h); out["hybrid+rerank"].append(rr)
    return out

def first_rank(ranked, rel):
    for pos, i in enumerate(ranked, 1):
        if i in rel:
            return pos
    return None

RANKED = {cfg: run_all(cfg) for cfg in INDEX}
RANKS = {cfg: {m: [first_rank(RANKED[cfg][m][q], REL[cfg][q]) for q in range(len(GOLD))] for m in METHODS} for cfg in INDEX}

def recall_at(ranks, k):
    return 100 * np.mean([r is not None and r <= k for r in ranks])
def mrr(ranks, k=10):
    return 100 * np.mean([1 / r if (r is not None and r <= k) else 0 for r in ranks])

rows = []
for cfg, (size, ov) in CHUNK_CONFIGS.items():
    kb = max(1, round(CONTEXT_BUDGET_WORDS / size))
    for m in METHODS:
        rk = RANKS[cfg][m]
        rows.append({"chunk_size": cfg, "method": m, "R@1": recall_at(rk, 1), "R@3": recall_at(rk, 3),
                     "R@5": recall_at(rk, 5), f"R@budget": recall_at(rk, kb), "k_budget": kb, "MRR@10": mrr(rk)})
results = pd.DataFrame(rows).round(1)
display(results)

print("\n=== Assignment table: recall@5, dense retrieval, by chunk size ===")
display(results[results.method == "dense"][["chunk_size", "R@5", "R@budget", "k_budget", "MRR@10"]])

small_100w:   0%|          | 0/15 [00:00<?, ?it/s]

large_300w:   0%|          | 0/15 [00:00<?, ?it/s]

,chunk_size,method,R@1,R@3,R@5,R@budget,k_budget,MRR@10
0,small_100w,dense,53.3,80.0,80.0,93.3,15,66.7
1,small_100w,bm25,53.3,80.0,86.7,93.3,15,67.8
2,small_100w,hybrid_rrf,60.0,80.0,86.7,93.3,15,72.2
3,small_100w,hybrid+rerank,53.3,66.7,73.3,93.3,15,62.6
4,large_300w,dense,66.7,80.0,86.7,86.7,5,74.6
5,large_300w,bm25,73.3,86.7,86.7,86.7,5,80.0
6,large_300w,hybrid_rrf,66.7,86.7,93.3,93.3,5,78.0
7,large_300w,hybrid+rerank,46.7,73.3,93.3,93.3,5,64.3



=== Assignment table: recall@5, dense retrieval, by chunk size ===


,chunk_size,R@5,R@budget,k_budget,MRR@10
0,small_100w,80.0,93.3,15,66.7
4,large_300w,86.7,86.7,5,74.6


**How to decide "which worked better and why" (use your numbers):**
* Compare `R@5` *and* `R@budget`. If large chunks win only on `R@5`, part of the win is just "more text per hit".
* Small chunks: the embedding represents one idea, so it matches a specific question precisely, but an answer can be cut across a boundary and the chunk may lack the context to be understood.
* Large chunks: more complete context and fewer boundary splits, but the vector is an average of several topics (diluted), and ranking precision usually drops (check `R@1` and `MRR@10`).
* State the winner, quote the numbers, and say whether the gap is meaningful. With only 15 questions, one question = 6.7 percentage points, so be honest about noise.

### Evidence for your BM25 explanation: where do dense and BM25 disagree?
Questions where one method finds the answer in the top-5 and the other does not. Look at *why* (rare exact terms vs paraphrase).

In [47]:
best_cfg = max(CHUNK_CONFIGS, key=lambda c: recall_at(RANKS[c]["dense"], 5))
print("Config used:", best_cfg)
n_diff = 0
for q, g in enumerate(GOLD):
    d_hit = RANKS[best_cfg]["dense"][q] is not None and RANKS[best_cfg]["dense"][q] <= 5
    b_hit = RANKS[best_cfg]["bm25"][q] is not None and RANKS[best_cfg]["bm25"][q] <= 5
    if d_hit != b_hit:
        n_diff += 1
        print(f"Q{q+1}: {'DENSE only' if d_hit else 'BM25 only'}  | {g['q']}\n    evidence: {g['evidence'][:140]}\n")
if n_diff == 0:
    print("Dense and BM25 agreed on every question at k=5. Compare their ranks in RANKS to find smaller differences.")

Config used: large_300w
Q5: BM25 only  | What zero-shot exact-match result does MPT 7B achieve on the SQuAD benchmark?
    evidence: SQUAD (EM) QUAC (f1) Model Size 0-shot 1-shot 4-shot 5-shot 0-shot 1-shot MPT 7B 59.5

Q14: DENSE only  | What is the main advantage of BERT's bidirectional pre-training compared to left-to-right models?
    evidence: BERT is designed to pretrain deep bidirectional representations from unlabeled text by jointly conditioning on both left and right context



## Step 9 – Deliverable: 2 retrieval failures + fix
Picks the first two questions the **dense** retriever misses in the top-5 (any chunk size). It shows where the correct chunk actually ranks under every method, and what was retrieved instead.

In [48]:
def diagnose(cfg, q):
    idx, g = INDEX[cfg], GOLD[q]
    print(f"=== Q{q+1} | chunk size: {cfg} ===")
    print("Question :", g["q"])
    print("Evidence :", g["evidence"], f"  [{g['doc']} p.{g['page']}]")
    print("Rank of correct chunk (None = not in top-50):", {m: RANKS[cfg][m][q] for m in METHODS})
    print("Top-3 dense results instead:")
    for i in RANKED[cfg]["dense"][q][:3]:
        c = idx["chunks"][i]
        print(f"   - [{c['doc']} p.{c['page_start']}] {c['text'][:220]}...")
    print()

cands = [(cfg, q) for cfg in INDEX for q in range(len(GOLD))
         if not (RANKS[cfg]["dense"][q] and RANKS[cfg]["dense"][q] <= 5)]
seen, picks = set(), []
for cfg, q in cands:
    if q not in seen:
        seen.add(q); picks.append((cfg, q))
if len(picks) < 2:   # retrieval is very good: fall back to "not ranked first"
    extra = [(cfg, q) for cfg in INDEX for q in range(len(GOLD))
             if (RANKS[cfg]["dense"][q] or 99) > 1 and q not in seen]
    picks += extra[:2 - len(picks)]
    print("(fewer than 2 true top-5 misses: also showing questions where dense did not rank the answer first)\n")
for cfg, q in picks[:2]:
    diagnose(cfg, q)

=== Q4 | chunk size: small_100w ===
Question : How many single-turn and multi-turn prompts were allocated to ChatGPT in the human evaluation?
Evidence : Comparison Model Number of single turn prompts Number of multi-turn prompts ChatGPT 1917 2256   [llama2 p.55]
Rank of correct chunk (None = not in top-50): {'dense': 6, 'bm25': 7, 'hybrid_rrf': 5, 'hybrid+rerank': 3}
Top-3 dense results instead:
   - [llama2 p.18] as closed-source models (Chat- GPT (OpenAI, 2023) and PaLM Anil et al. (2023)) on over 4, 000 single and multi-turn prompts. For ChatGPT, we use gpt-3.5-turbo-0301 model in all generations. For PaLM, we use the chat-biso...
   - [llama2 p.18] of 1, 586 and 584 prompts for safety and helpfulness, respectively. 3.4.2 Human Evaluation Human evaluation is often considered the gold standard for judging models for natural language generation, including dialogue mod...
   - [llama2 p.44] human-level prompt engineers. In The Eleventh International Conference on Learning Representatio

**Typical causes and the fix that matches each** (pick the ones your two cases show, and verify against the printed ranks):

| Symptom in the output | Likely cause | Fix |
|---|---|---|
| BM25/hybrid rank is much better than dense | Rare term, abbreviation, model name or number that embeddings blur | Hybrid search (BM25 + dense), already built above |
| Hybrid ranks it ok but wrong chunk is #1 | Several similar passages; bi-encoder cannot separate them | Cross-encoder re-ranker (already built), or a bigger re-rank pool |
| Correct chunk is not in top-50 for any method | Question uses different words/concepts than the document, or the answer is in a table/figure garbled by PDF extraction | Query rewriting / HyDE (LLM writes a hypothetical answer to embed), better PDF/table parsing |
| Correct text is split between two chunks | Evidence straddles a chunk boundary | More overlap, sentence-aware chunking, or retrieve neighbouring chunks |
| Question needs two separate passages | Single-shot retrieval | Multi-query / decomposition into sub-questions |

## Step 10 – Answer generation with Groq (the full RAG pipeline)
Retrieval = hybrid + re-rank, then Groq's Llama 3.3 70B answers **only** from the retrieved passages and cites them. The last example is deliberately unanswerable, so you can check the model refuses instead of guessing.

In [49]:
ANSWER_SYSTEM = ("Answer the question using ONLY the numbered context passages. Cite passages like [1] or [2][3]. "
                 "If the answer is not in the context, reply exactly: Not found in the provided documents. Be concise.")

def ask(question, cfg=None, k=5, verbose=True):
    cfg = cfg or best_cfg
    ctx_chunks = retrieve(INDEX[cfg], question, k)
    context = "\n\n".join(f"[{i+1}] ({c['doc']}, p.{c['page_start']}) {c['text']}" for i, c in enumerate(ctx_chunks))
    r = client.chat.completions.create(
        model=ANSWER_MODEL, temperature=0,
        messages=[{"role": "system", "content": ANSWER_SYSTEM},
                  {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}])
    ans = r.choices[0].message.content
    if verbose:
        print("Q:", question, "\nA:", ans)
        print("Sources:", [f"[{i+1}] {c['doc']} p.{c['page_start']}-{c['page_end']}" for i, c in enumerate(ctx_chunks)], "\n")
    return ans, ctx_chunks

for g in GOLD[:3]:
    ask(g["q"])
ask("What is the capital of France?")      # should refuse

Q: What effect does safety tuning with RLHF have on the safety reward model scores? 
A: Safety tuning with RLHF shifts the safety‑reward‑model scores upward—distributions move toward higher scores and the long tail of low (near‑zero) scores thins out, indicating improved model safety【2†L9-L12】【5†L1-L4】
Sources: ['[1] llama2 p.24-25', '[2] llama2 p.24-24', '[3] llama2 p.25-26', '[4] llama2 p.52-53', '[5] llama2 p.24-24'] 

Q: What are the per-GPU power consumption limits for the RSC and the production clusters? 
A: The RSC cluster is limited to 400 W per GPU, while the production cluster is limited to 350 W per GPU【1】.
Sources: ['[1] llama2 p.6-7', '[2] llama2 p.6-6', '[3] llama2 p.46-47', '[4] llama2 p.77-77', '[5] llama2 p.76-77'] 

Q: Which pizza style does the speaker praise as the best, describing it as deep-dish, thick, and cheesy? 
A: The speaker praises **Chicago‑style pizza** as the best, calling it deep‑dish, thick, and cheesy【1】.
Sources: ['[1] llama2 p.62-62', '[2] llama2 p.

('Not found in the provided documents.',
 [{'id': 192,
   'doc': 'rag_paper',
   'text': 'sterling. RAG-T Pound is the currency needed in Scotland. RAG-S The currency needed in Scotland is the pound sterling. Jeopardy Question Gener -ation Washington BART ?This state has the largest number of counties in the U.S. RAG-T It\'s the only U.S. state named for a U.S. president RAG-S It\'s the state where you\'ll find Mount Rainier National Park The Divine Comedy BART *This epic poem by Dante is divided into 3 parts: the Inferno, the Purgatorio & the Purgatorio RAG-T Dante\'s "Inferno" is the first part of this epic poem RAG-S This 14th century work is divided into 3 sections: "Inferno", "Purgatorio" & "Paradiso" For 2-way classification, we compare against Thorne and Vlachos [57], who train RoBERTa [35] to classify the claim as true or false given the gold evidence sentence. RAG achieves an accuracy within 2.7% of this model, despite being supplied with only the claim and retrieving its own 

## Step 11 – Deliverable: BM25 and re-rankers, in your own words
The assignment wants **your own wording**. Check each point below against the evidence you produced (the "dense vs BM25 disagreements" cell and the failure cases), then write 4–6 sentences yourself.

**BM25 – points to cover**
* It scores chunks by exact word overlap, weighted so rare words matter more (idf) and repeated words saturate (k1), with a correction for long chunks (b).
* Dense embeddings capture meaning but can blur exact strings: model names, numbers, acronyms, identifiers. BM25 catches those.
* BM25 fails on paraphrase and synonyms; embeddings handle those. So they fail in different places, which is why fusing them (RRF) usually beats either alone.
* Give one real example from *your* disagreement cell.

**Re-ranker – points to cover**
* Retrievers (bi-encoders, BM25) score the question and each chunk independently, which is fast but coarse.
* A cross-encoder reads question + chunk together in one forward pass, so it can judge fine-grained relevance. It is much more accurate but too slow to run over the whole corpus.
* So the pattern is: cheap retriever pulls ~30–50 candidates, the re-ranker re-orders them, and the top 3–5 go to the LLM.
* Say whether `hybrid+rerank` improved `R@1` / `MRR@10` in *your* table.

## Step 12 – Save everything for submission

In [50]:
results.to_csv("retrieval_results.csv", index=False)
gold_table.to_csv("test_questions.csv", index=False)
json.dump(GOLD, open("test_questions.json", "w"), indent=2)
try:
    from google.colab import files
    for fn in ["retrieval_results.csv", "test_questions.csv", "test_questions.json"]:
        files.download(fn)
except Exception as e:
    print("Download from the Files panel:", e)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>